In [1]:
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import time
import os
from dotenv import load_dotenv


In [2]:
load_dotenv()
API_KEY = os.getenv("API_KEY")
ytb = build("youtube", "v3", developerKey=API_KEY)

In [3]:
def searchByKeyword(keyword, region_code='US', max_results=30):
    request = ytb.search().list(
        q = keyword,
        
        part = 'snippet',
        type = 'video',
        order = 'relevance',
        regionCode = region_code,
        maxResults = max_results
    )
    response = request.execute()
    videos = response.get('items', [])
    results = []
    for video in videos:
        channel_title = video['snippet']['channelTitle']
        video_id = video['id']['videoId']
        title = video['snippet']['title']
        description = video['snippet']['description']
        published_at = video['snippet']['publishedAt']
        results.append({
            'channel_title': channel_title,
            'video_id': video_id,
            'title': title,
            'description': description,
            'published_at': published_at
        })
    return results

In [4]:
def getComments(video_id, max_results=5):
    try:    
        request = ytb.commentThreads().list(
            part='snippet',
            videoId=video_id,
            textFormat='plainText',
            maxResults = max_results,
            order='time'
        )
        response = request.execute()
        comments = response.get('items', [])
        comment_texts = []
        for comment in comments:
            text = comment['snippet']['topLevelComment']['snippet']['textDisplay']
            author = comment['snippet']['topLevelComment']['snippet']['authorDisplayName']
            published_at = comment['snippet']['topLevelComment']['snippet']['publishedAt']
            comment_texts.append({
                'text': text,
                'author': author,
                'published_at': published_at
            })
        return comment_texts
    except HttpError as e:
        if e.resp.status == 403:
            print(f"Comments are disabled for video {video_id}. Skipping...")
            return []
        else:
            raise

In [5]:
def getStatitics(video_id):
    request = ytb.videos().list(
        part='statistics',
        id=video_id
    )
    response = request.execute()
    items = response.get('items', [])
    if not items:
        return {}
    stats = items[0].get('statistics', {})
    return {
        'viewCount': stats.get('viewCount', '0'),
        'likeCount': stats.get('likeCount', '0'),
        'dislikeCount': stats.get('dislikeCount', '0'),
        'commentCount': stats.get('commentCount', '0')
    }

In [6]:
def main(keyword='truc tiep game', max_results=5):
    all_data = []
    videos = searchByKeyword(keyword)
    for video in videos:
        video_id = video['video_id']
        comments = getComments(video_id)
        stats = getStatitics(video_id)
        video_data = {
            'video_id': video_id,
            'channel_title': video['channel_title'],
            'title': video['title'],
            'description': video['description'],
            'published_at': video['published_at'],
            'comments': comments,
            'statistics': stats
        }
        all_data.append(video_data)
        
    output_dir = 'json_file'

    filename = f"{keyword.replace(' ', '_')}_data.json"
    filepath = os.path.join(output_dir, filename)

    with open(filepath, 'w', encoding='utf-8') as f:
        import json
        json.dump(all_data, f, ensure_ascii=False, indent=4)
        return filepath
    print(f"Data saved to {filepath}")


In [7]:
if __name__ == "__main__":
    main()